In [91]:
import pandas as pd
rentals_raw = pd.read_csv("data/bike_rentals.csv")

rentals = rentals_raw

print(rentals.shape)
print(rentals.dtypes)

(403, 15)
rental_id            str
station              str
bike_type            str
helmets_text         str
is_member            str
insured              str
paid_online          str
status               str
coupon_code      float64
damage_notes     float64
late_fee_note    float64
start_date           str
end_date             str
price                str
rating           float64
dtype: object


In [92]:
print(rentals["price"].dtype)

str


In [93]:
empty_colums = rentals.columns[rentals.isna().all()].to_list()
empty_colums

['coupon_code', 'damage_notes', 'late_fee_note']

In [94]:
rentals = rentals.drop(columns=empty_colums)


In [95]:
print("empty/ removed colums", len(empty_colums))
print("colums left:", rentals.shape[1])
print(rentals.shape)

empty/ removed colums 3
colums left: 12
(403, 12)


In [96]:
rentals['price'].head()

0      SEK 855.50
1    SEK 1,311.50
2      SEK 350.00
3      SEK 569.50
4      SEK 828.00
Name: price, dtype: str

In [97]:
rentals['price'] = rentals['price'].str.replace('SEK', "").str.strip() 
print(rentals['price'].head(3))

0      855.50
1    1,311.50
2      350.00
Name: price, dtype: str


In [98]:
rentals['price'] = rentals['price'].str.replace(',', "").str.strip() 
print(rentals['price'].head(3))

0     855.50
1    1311.50
2     350.00
Name: price, dtype: str


In [99]:
rentals['price'] = rentals['price'].astype(float)
print(rentals['price'].head(3))

0     855.5
1    1311.5
2     350.0
Name: price, dtype: float64


In [100]:
rentals.describe()

,price,rating
count,352.000000,288.000000
mean,847.348011,4.225694
std,633.435513,1.019547
min,154.500000,1.000000
25%,406.000000,4.000000
50%,668.250000,5.000000
75%,1194.625000,5.000000
max,3401.000000,5.000000


In [101]:
most_expensive_rentals = rentals.nlargest(3, 'price')[['rental_id', 'bike_type', 'status', 'price']]
most_expensive_rentals

,rental_id,bike_type,status,price
187,R-5282,Cargo bike,Completed,3401.0
49,R-5209,Cargo bike,Completed,2979.5
270,R-5378,E-bike,Completed,2941.0


In [102]:
missing_count = rentals['price'].isna().sum()
missing_percent = rentals['price'].isna().mean()*100

print(missing_count)
print(missing_percent)

51
12.655086848635236


In [103]:
Compare = pd.crosstab(
    rentals["price"].isna(),
    rentals["status"].str.lower() == 'cancelled',
    rownames = ["missing_price"],
    colnames = ["cancelled"]
)
Compare

cancelled,False,True
missing_price,,
False,352,0
True,4,47


there is 4 missing price that was not cancelled
One solution could be not to include dose 4 in the analysis, because we don't know what the price should be and can't guess.

In [104]:
bool_columns = [
    col for col in rentals.columns
    if set(rentals[col].dropna().unique()).issubset({'t', 'f'})           
]
print("columns with only t and f:", bool_columns) 


columns with only t and f: ['is_member', 'insured', 'paid_online']


status dont have true or false, it use text instead of bool , so its not in the list bool_columns

In [105]:
for col in bool_columns:
    rentals[col] =rentals[col].map({'t': True, 'f': False})



In [106]:
rentals[bool_columns].isna().sum()   

is_member      0
insured        3
paid_online    0
dtype: int64

In [107]:
summary = rentals.groupby('is_member')['price'].agg(
    median_price= 'median',
    rental_count='count'
)
summary

,median_price,rental_count
is_member,,
False,756.25,248
True,558.00,104


In [108]:
rentals['is_member'].value_counts()

is_member
False    288
True     115
Name: count, dtype: int64

In [109]:
print(rentals[['start_date', 'end_date']].isna().sum())

start_date    0
end_date      0
dtype: int64


In [110]:
rentals['start_date'] = pd.to_datetime(
    rentals['start_date'],
    errors='coerce')
rentals['end_date'] = pd.to_datetime(
    rentals['end_date'],
    errors='coerce')

In [111]:
print('after',rentals[['start_date', 'end_date']].isna().sum())

after start_date    2
end_date      0
dtype: int64


In [112]:
print(rentals[['start_date', 'end_date']].dtypes)

start_date    datetime64[us]
end_date      datetime64[us]
dtype: object


In [113]:
rentals['rental_days'] = (rentals['end_date'] - rentals['start_date']).dt.days
print(rentals['rental_days'].describe())

count    401.000000
mean       2.955112
std        1.835206
min       -2.000000
25%        1.000000
50%        3.000000
75%        4.000000
max        7.000000
Name: rental_days, dtype: float64


min    is   -2.000000

In [114]:
missing_ratings = rentals['rating'].isna().sum()
status = rentals['status'].isna().sum()
print(missing_ratings)
print(status)

ratings = pd.crosstab(
    rentals['rating'].isna(),
    rentals['status'],
    rownames=['rating_missing'],
    colnames=['status']
)
ratings

115
0


status,Cancelled,Completed,No-show
rating_missing,,,
False,0,288,0
True,47,46,22


In [115]:
rentals['has_rating'] =rentals['rating'].notna()
rentals['has_rating'].value_counts()

has_rating
True     288
False    115
Name: count, dtype: int64

In [116]:
print(rentals['rating'].describe())
print("median", rentals['rating'].median())

count    288.000000
mean       4.225694
std        1.019547
min        1.000000
25%        4.000000
50%        5.000000
75%        5.000000
max        5.000000
Name: rating, dtype: float64
median 5.0


In [137]:
rentals['helmets_text'].value_counts(dropna=False)

helmets_text
1 helmet          146
2 helmets         127
no helmets         59
1 child helmet     47
3 helmets          20
NaN                 4
Name: count, dtype: int64

In [120]:
rentals['helmets_text'] = rentals['helmets_text'].str.lower()
rentals['helmets_text'].value_counts(dropna=False)

helmets_text
1 helmet          146
2 helmets         127
no helmets         59
1 child helmet     47
3 helmets          20
NaN                 4
Name: count, dtype: int64

In [140]:
rentals['child_helmet'] = rentals['helmets_text'].str.contains('child')
rentals['child_helmet'].value_counts()

child_helmet
False    356
True      47
Name: count, dtype: int64

In [139]:
rentals['helmets_count'] = rentals['helmets_text'].str.extract(r'([\d.]+)', expand=False).astype(float)

rentals['helmets_count'].value_counts()



helmets_count
1.0    193
2.0    127
3.0     20
Name: count, dtype: int64

In [156]:
is_no_helmets = rentals['helmets_text'].str.contains('no helmets')
rentals['helmets_count'] = rentals['helmets_count'].where(~is_no_helmets, 0)
print(rentals['helmets_count'].value_counts(dropna=False).sort_index())
print(rentals['helmets_count'].isna().sum())

helmets_count
0.0     59
1.0    193
2.0    127
3.0     20
NaN      4
Name: count, dtype: int64
4


In [ ]:
cols = ['station', 'bike_type', 'status']

for col in cols:
    raw_unique = rentals[col].nunique()
    clean_unique = rentals[col].str.strip().str.lower().nunique()
    print(f"{col}: raw = {raw_unique}, cleaned = {clean_unique}")

station: raw = 20, cleaned = 5
bike_type: raw = 3, cleaned = 3
status: raw = 3, cleaned = 3


station is inconsistent : raw = 20, cleaned = 5

In [160]:
rentals['station'] = rentals['station'].str.strip().str.title()
print(rentals['station'].value_counts())

station
Central        107
Södermalm       95
Kungsholmen     89
Östermalm       57
Hammarby        55
Name: count, dtype: int64


In [162]:
cols = ['station', 'bike_type', 'status']

for col in cols:
    raw_unique = rentals[col].nunique()
    clean_unique = rentals[col].str.strip().str.lower().nunique()
    print(f"{col}: raw = {raw_unique}, cleaned = {clean_unique}")
# fixed

station: raw = 5, cleaned = 5
bike_type: raw = 3, cleaned = 3
status: raw = 3, cleaned = 3


In [163]:
print(rentals.duplicated().sum())

3


In [164]:
print(rentals.duplicated(subset=['rental_id']).sum())

3


In [166]:
rentals = rentals.drop_duplicates()
print(rentals.duplicated().sum())
print(rentals.duplicated(subset=['rental_id']).sum())

0
0


In [168]:
print(rentals.shape)

(400, 16)
